# Mikroskopi karelerinde hücre segmentasyonu ve sayımı

Üsküdar Üniversitesi · Bilgisayar Mühendisliği Yüksek Lisans · Sayısal Görüntü İşleme

Bu defter **29033** tohumu ile kareleri yeniden seçer, Sekans 01'de 30 kontrollü
deney ve B00 başlangıç yöntemini çalıştırır, üç adayı doğrular ve Sekans 02 için
referanslardan bağımsız tahminler üretir. Veri setinin eğitim arşivi kullanılır.
Her defter oturumu yeniden üretim sonuçlarını ayrı bir geçici klasörde tutar.

## 1. Ortam ve eğitim arşivi

Yerel Jupyter için defteri depo kökünden açın ve `requirements.txt` içeriğini
aktif Python 3.12 ortamına yükleyin. Colab'da önce kaynak dosyaları oturuma
klonlayın ve `%cd /content/DEPO` ile depo klasörüne geçin. Aşağıdaki kurulum
satırının başındaki `#` işaretini yalnızca paket yüklemeniz gerekiyorsa kaldırın.

ZIP dosyası varsayılan olarak depo altındaki `data/` içinde aranır. Farklı
bir ZIP veya açılmış eğitim klasörü için `DATA_OVERRIDE` değerini değiştirin.
Herhangi bir arşiv yoksa resmî CTC eğitim ZIP'i geçici klasöre indirilir.

In [ ]:
# %pip install -r requirements.txt

In [ ]:
import csv
import json
import os
from html import escape
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
from urllib.request import urlopen
from zipfile import is_zipfile

from IPython.display import HTML, Image, display

ROOT = Path.cwd().resolve()
if not (ROOT / "cellcount" / "run.py").is_file():
    raise FileNotFoundError("Önce çalışma dizinini kaynak deponun köküne ayarlayın.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cellcount.data import ARCHIVE_URL, DEFAULT_SEED

DATA_OVERRIDE = ""
override = DATA_OVERRIDE or os.environ.get("GOWT1_DATA")
WORK = Path(tempfile.mkdtemp(prefix="gowt1-")).resolve()
RESULTS = WORK / "results"
FIGURES = WORK / "private_figures"
DATA = Path(override).expanduser().resolve() if override else ROOT / "data" / "Fluo-N2DH-GOWT1.zip"
if not DATA.exists():
    if override:
        raise FileNotFoundError(DATA)
    DATA = WORK / "Fluo-N2DH-GOWT1.zip"
    print("Resmî CTC eğitim ZIP'i indiriliyor.")
    with urlopen(ARCHIVE_URL, timeout=60) as response, DATA.open("wb") as stream:
        shutil.copyfileobj(response, stream)
    if not is_zipfile(DATA):
        raise ValueError("İndirilen dosya geçerli bir ZIP arşivi olarak açılamadı.")

def run(module, *arguments):
    command = [sys.executable, "-m", module, *(str(value) for value in arguments)]
    environment = dict(os.environ, PYTHONUTF8="1")
    completed = subprocess.run(command, cwd=ROOT, env=environment,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               encoding="utf-8")
    if completed.stdout:
        print(completed.stdout, end="")
    completed.check_returncode()

def read(path):
    with path.open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))

def show_table(rows, keys):
    headers = "".join(f"<th>{escape(key)}</th>" for key in keys)
    lines = ["<tr>" + "".join(f"<td>{escape(str(row[key]))}</td>" for key in keys)
             + "</tr>" for row in rows]
    display(HTML("<table><thead><tr>" + headers + "</tr></thead><tbody>"
                 + "".join(lines) + "</tbody></table>"))

print("Veri:", DATA)
print("Deney klasörü:", RESULTS)

## 2. Kareleri seç

Sekans 01'den önce 5 doğrulama, sonra 10 geliştirme karesi seçilir.
Her geliştirme karesi ile her doğrulama karesi arasında en az beş
kullanılmayan kare bulunur. Sekans 02'den birbirinden en az beş kareyle
ayrılan üç ardışık beş karelik blok seçilir.

In [ ]:
run("cellcount.run", "select", "--data", DATA, "--out", RESULTS, "--seed", DEFAULT_SEED)
selection = read(RESULTS / "selected_frames.csv")
assert selection == read(ROOT / "selected_frames.csv")
show_table(selection, ("group", "sequence", "block", "frame", "filename", "seed"))

## 3. Kontrollü deneyleri çalıştır ve adayları doğrula

Geliştirme karşılaştırmalarında 10 karenin TRA referans sayıları kullanılır.
Sabit, Otsu ve adaptif eşik ailelerinden birer aday, geliştirme MAE'si
temel alınarak seçilir. Üç aday 5 doğrulama karesinde karşılaştırılır.
Nihai seçimde MAE önceliklidir. Eşitlik çözümünde RMSE, süre ve etkin
işlem sayısı kullanılır.

In [ ]:
run("cellcount.run", "develop", "--data", DATA, "--out", RESULTS)
experiments = read(RESULTS / "experiments.csv")
show_table(experiments, ("experiment", "intensity", "filter", "segmentation",
                         "morphology", "mae", "rmse"))
validation = read(RESULTS / "validation_results.csv")
show_table(validation, ("rank", "experiment", "family", "mae", "rmse",
                        "mean_ms", "selected"))

## 4. Test tahminlerini referansları okumadan kaydet

E24, doğrulamada seçilen 5×5 Gaussian → adaptif Gaussian eşik
(121×121, C = −2) → 5×5 eliptik açma ve kapama → delik doldurma
→ 8 bağlantılı bileşen ve 250 px alan sınırı zinciridir.
Kilit aşaması yalnızca 15 ham test karesini işler ve her iki yöntemin
tahminini yazar. SHA-256 özetleri seçim, tahmin ve işlem kodunu sabitler.

In [ ]:
run("cellcount.run", "lock", "--data", DATA, "--out", RESULTS)
manifest = json.loads((RESULTS / "final_pipeline.json").read_text(encoding="utf-8"))
print("Nihai yöntem:", manifest["final"]["id"])
print("Tahmin dosyası SHA-256:", manifest["sha256"]["test_predictions_locked.csv"])
predictions = read(RESULTS / "test_predictions_locked.csv")
show_table(predictions, ("block", "frame", "baseline", "final"))

## 5. Ara işlem görsellerini incele

Örnek mikroskopi panelleri ve referanssız test eğrileri geçici klasöre yazılır.
Defter GitHub'a gönderilecekse kaydetmeden önce tüm çıktı hücrelerini temizleyin.

In [ ]:
run("cellcount.figures", "--data", DATA, "--results", RESULTS, "--output", FIGURES)
for filename in ("fig02_intensity.png", "fig04_steps.png", "fig05_temporal.png"):
    display(Image(filename=str(FIGURES / filename), width=850))
print("Özel görseller:", FIGURES)

## 6. GitHub kilidinden sonra bağımsız değerlendirme

Bu hücre varsayılan olarak devre dışıdır. Önce kayıtlı tahmin commit'ini
GitHub'a gönderin ve README'deki değerlendirme adımlarını izleyin.
Aşağıda **uzak dalda bulunan özgün kilit commit'inin SHA değerini** girip
açıkça etkinleştirdiğinizde doğrulama geçerse araç referansları okur ve
depo kökünde test CSV'lerini üretir. Sonuçları kilit commit'inden ayrı
bir commit ile gönderin. Geçici deneme tahminlerini depo köküne kopyalamayın.

In [ ]:
RUN_EVALUATION = False
LOCK_COMMIT = ""
LOCK_BRANCH = "main"
if RUN_EVALUATION:
    if not LOCK_COMMIT:
        raise ValueError("Önce GitHub kilit commit SHA değerini belirtin.")
    run("cellcount.run", "evaluate", "--data", DATA, "--out", ROOT,
        "--lock-commit", LOCK_COMMIT, "--branch", LOCK_BRANCH)
    show_table(read(ROOT / "test_metrics.csv"),
               ("method", "mae", "rmse", "bias", "exact_accuracy", "mean_ms"))
else:
    print("Bağımsız değerlendirme hücresi devre dışı.")